# 02_TF-IDF_Content_Based_Recommendation.ipynb

基于 TF-IDF + UGC 的美食推荐系统实验

> **理论依据**：《推荐系统算法详解》课件 + `02_tf-idf_note.md`  
> **参考代码**：`6_tfidf代码实现.ipynb` - 手动实现 TF-IDF 流程  
> **数据集**：Yelp Academic Dataset (archive_4)  
> **实验目标**：理解 TF-IDF 算法原理，实现基于内容的推荐系统

---

## 实验结构

| 阶段   | 内容                          | 步骤       |
| ------ | ----------------------------- | ---------- |
| 阶段一 | 环境准备与数据加载            | Step 1-4   |
| 阶段二 | TF-IDF 手动实现（教学目的）   | Step 5-10  |
| 阶段三 | 使用 scikit-learn 实现 TF-IDF | Step 11-14 |
| 阶段四 | 构建餐厅特征向量              | Step 15-17 |
| 阶段五 | 构建用户画像                  | Step 18-20 |
| 阶段六 | 基于内容的推荐                | Step 21-24 |
| 阶段七 | 评估与可视化                  | Step 25-27 |
| 阶段八 | 扩展实验（可选）              | Step 28-30 |


---

# 阶段一：环境准备与数据加载


## Step 1: 导入依赖库


In [ ]:
# 基础数据处理库
import pandas as pd
import numpy as np
import json
import math
import warnings
from pathlib import Path
from collections import Counter

# TF-IDF相关库
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

# 可视化库
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# 文本处理
import re
from functools import lru_cache

# 设置显示选项
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
sns.set_style('whitegrid')

print("✅ 依赖库导入完成！")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"scikit-learn: {__import__('sklearn').__version__}")

## Step 2: 定义数据加载工具函数


In [ ]:
# 数据集路径配置
DATA_DIR = Path('../data/archive_4')


def load_json_lines(filepath, nrows=None, sample_ratio=None):
    """
    加载JSON Lines格式文件

    Args:
        filepath: JSON文件路径
        nrows: 加载前n行（用于快速测试）
        sample_ratio: 采样比例（0-1之间），随机采样部分数据

    Returns:
        DataFrame: 加载的数据
    """
    data = []

    # 先计算总行数（如果需要采样）
    if sample_ratio is not None:
        print(f"📊 计算文件总行数以进行采样...")
        total_lines = sum(1 for _ in open(filepath, 'r', encoding='utf-8'))
        nrows = int(total_lines * sample_ratio)
        print(f"   总行数: {total_lines}, 采样行数: {nrows}")
        # 生成随机采样索引
        sample_indices = set(np.random.choice(
            total_lines, nrows, replace=False))
    else:
        sample_indices = None

    with open(filepath, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            # 如果指定了nrows且没有采样，则限制行数
            if nrows and sample_ratio is None and i >= nrows:
                break

            # 如果采样，检查是否在采样索引中
            if sample_indices is not None and i not in sample_indices:
                continue

            try:
                data.append(json.loads(line))
            except json.JSONDecodeError:
                continue

    return pd.DataFrame(data)


def filter_food_businesses(df):
    # 必须包含 Restaurants 或Cafe
    must_have = r'Restaurants|Cafe|Cafes'
    # 排除纯零售
    exclude = r'Grocery|Pharmacy|Department Store|Pet Food'
    
    mask = (
        df['categories'].notna() &
        df['categories'].str.contains(must_have, case=False, na=False) &
        ~df['categories'].str.contains(exclude, case=False, na=False)
    )
    return df[mask].copy()


print("✅ 数据加载工具函数定义完成！")
print(f"数据目录: {DATA_DIR.absolute()}")

## Step 3: 加载 Yelp 数据集


In [ ]:
# 加载商家数据（10k条）
print("\n" + "="*80)
print("📍 加载商家数据 (business.json)")
print("="*80)

business_df = load_json_lines(
    DATA_DIR / 'yelp_academic_dataset_business.json', nrows=10000)
print(f"总商家数: {len(business_df)}")
print(f"字段: {list(business_df.columns)}")

# 筛选食品相关商家
food_business_df = filter_food_businesses(business_df)
print(
    f"\n🍽️ 食品相关商家: {len(food_business_df)} / {len(business_df)} ({len(food_business_df)/len(business_df)*100:.1f}%)")

# 查看食品商家样本
display(food_business_df[['business_id', 'name',
        'categories', 'stars', 'review_count']].head())

In [ ]:
# 加载评论数据（100k条）
print("\n" + "="*80)
print("💬 加载评论数据 (review.json)")
print("="*80)

review_df = load_json_lines(
    DATA_DIR / 'yelp_academic_dataset_review.json', nrows=100000)
print(f"总评论数: {len(review_df)}")
print(f"字段: {list(review_df.columns)}")

# 只保留食品商家的评论
food_business_ids = set(food_business_df['business_id'])
food_reviews_df = review_df[review_df['business_id'].isin(
    food_business_ids)].copy()
print(
    f"\n📝 食品商家相关评论: {len(food_reviews_df)} / {len(review_df)} ({len(food_reviews_df)/len(review_df)*100:.1f}%)")

display(food_reviews_df[['review_id', 'user_id',
        'business_id', 'stars', 'text']].head(2))

In [ ]:
# 检查有多少食品商家实际有评论
covered = food_business_df['business_id'].isin(
    food_reviews_df['business_id'].unique()).sum()
print(f"有评论的食品商家: {covered} / {len(food_business_df)}")
print(f"评论覆盖的独立商家数: {food_reviews_df['business_id'].nunique()}")

In [ ]:
# 加载用户数据（可选，用于用户画像分析）
print("\n" + "="*80)
print("👥 加载用户数据 (user.json)")
print("="*80)

user_df = load_json_lines(
    DATA_DIR / 'yelp_academic_dataset_user.json', nrows=10000)
print(f"总用户数: {len(user_df)}")
print(f"字段: {list(user_df.columns)}")

# 只保留有评论的用户
active_user_ids = set(food_reviews_df['user_id'])
active_users_df = user_df[user_df['user_id'].isin(active_user_ids)].copy()
print(f"\n👤 活跃用户: {len(active_users_df)} / {len(user_df)}")

## Step 4: 数据探索


In [ ]:
# 4.1 评论文本长度分布
print("\n" + "="*80)
print("📝 评论文本长度分析")
print("="*80)

food_reviews_df['text_length'] = food_reviews_df['text'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 长度分布直方图
axes[0].hist(food_reviews_df['text_length'], bins=50,
             color='skyblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Review Length (characters)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Review Text Length')
axes[0].axvline(food_reviews_df['text_length'].mean(), color='red',
                linestyle='--', label=f"Mean: {food_reviews_df['text_length'].mean():.0f}")
axes[0].axvline(food_reviews_df['text_length'].median(), color='green',
                linestyle='--', label=f"Median: {food_reviews_df['text_length'].median():.0f}")
axes[0].legend()

# 评分分布
food_reviews_df['stars'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color='gold', edgecolor='black')
axes[1].set_xlabel('Stars')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Review Ratings')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print(f"\n📊 评论文本统计:")
print(food_reviews_df['text_length'].describe())

In [ ]:
# 4.2 商家类别分布
print("\n" + "="*80)
print("🍽️ 商家类别分析")
print("="*80)

# 提取所有类别并统计频率
all_categories = []
for cats in food_business_df['categories'].dropna():
    all_categories.extend([c.strip() for c in cats.split(',')])

category_counts = Counter(all_categories)
top_categories = category_counts.most_common(15)

print(f"\n🔝 Top 15 商家类别:")
for cat, count in top_categories:
    print(f"  {cat}: {count}")

# 可视化
fig, ax = plt.subplots(figsize=(10, 6))
cats, counts = zip(*top_categories)
ax.barh(cats, counts, color='coral')
ax.set_xlabel('Count')
ax.set_title('Top 15 Business Categories')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# 4.3 查看示例评论
print("\n" + "="*80)
print("💬 示例评论文本")
print("="*80)

sample_reviews = food_reviews_df.sample(3, random_state=42)
for idx, row in sample_reviews.iterrows():
    business_name = food_business_df[food_business_df['business_id']
                                     == row['business_id']]['name'].values
    business_name = business_name[0] if len(business_name) > 0 else 'Unknown'

    print(f"\n{'='*60}")
    print(f"🍴 商家: {business_name}")
    print(f"⭐ 评分: {row['stars']} stars")
    print(f"📝 评论: {row['text'][:300]}...")

---

# 阶段二：TF-IDF 算法手动实现（教学目的）

本阶段将手动实现 TF-IDF 算法，帮助理解其原理。参考：`6_tfidf代码实现.ipynb`


## Step 5: 准备示例数据


In [ ]:
# 5.1 使用老师示例的基础数据
docA = "The cat sat on my bed"
docB = "The dog sat on my knees"

print("="*80)
print("📚 基础示例文档（老师示例）")
print("="*80)
print(f"文档A: {docA}")
print(f"文档B: {docB}")

# 5.2 使用真实Yelp评论作为示例
print("\n" + "="*80)
print("🍽️ 真实Yelp评论示例")
print("="*80)

# 选择3条不同商家的评论
sample_biz_ids = food_reviews_df['business_id'].unique()[:3]
yelp_docs = []
for bid in sample_biz_ids:
    review_text = food_reviews_df[food_reviews_df['business_id']
                                  == bid]['text'].iloc[0]
    # 只取前100个字符作为简单示例
    yelp_docs.append(review_text[:100])
    business_name = food_business_df[food_business_df['business_id']
                                     == bid]['name'].values[0]
    print(f"\n商家: {business_name}")
    print(f"评论: {review_text[:100]}...")

# 合并为文档列表
documents = [docA, docB] + yelp_docs
print(f"\n✅ 共准备 {len(documents)} 个示例文档")

## Step 6: 文本预处理


In [ ]:
# 6.1 分词（Tokenization）
print("="*80)
print("🔤 Step 6.1: 分词（Tokenization）")
print("="*80)

# 使用简单split分词（小写化）
bowA = docA.lower().split()
bowB = docB.lower().split()

print(f"文档A词袋: {bowA}")
print(f"文档B词袋: {bowB}")

# 6.2 构建词表（Vocabulary）
print("\n" + "="*80)
print("📖 Step 6.2: 构建词表（Vocabulary）")
print("="*80)

wordSet = set(bowA).union(set(bowB))
print(f"词表大小: {len(wordSet)}")
print(f"词表: {sorted(wordSet)}")

In [ ]:
# 6.3 构建词频统计字典
print("\n" + "="*80)
print("📊 Step 6.3: 构建词频统计字典")
print("="*80)

# 初始化词频字典
wordDictA = dict.fromkeys(wordSet, 0)
wordDictB = dict.fromkeys(wordSet, 0)

# 统计词频
for word in bowA:
    wordDictA[word] += 1
for word in bowB:
    wordDictB[word] += 1

# 转换为DataFrame展示
word_freq_df = pd.DataFrame([wordDictA, wordDictB], index=['文档A', '文档B'])
print("\n词频统计矩阵:")
display(word_freq_df)

print("\n💡 观察：")
print("  - 'the', 'sat', 'on', 'my' 在两个文档中都出现")
print("  - 'cat', 'bed' 只在文档A中出现")
print("  - 'dog', 'knees' 只在文档B中出现")

## Step 7: 计算 TF（词频 Term Frequency）


In [ ]:
print("="*80)
print("📈 Step 7: 计算词频 (TF)")
print("="*80)


def computeTF(wordDict, bow):
    """
    计算词频 (Term Frequency)

    公式: TF(t,d) = (词t在文档d中出现的次数) / (文档d的总词数)

    Args:
        wordDict: 词频字典
        bow: 文档词袋

    Returns:
        tfDict: 每个词的TF值
    """
    tfDict = {}
    bowCount = len(bow)

    for word, count in wordDict.items():
        tfDict[word] = count / bowCount

    return tfDict


# 计算两个文档的TF
tfA = computeTF(wordDictA, bowA)
tfB = computeTF(wordDictB, bowB)

# 展示结果
tf_df = pd.DataFrame([tfA, tfB], index=['文档A TF', '文档B TF'])
print("\nTF值矩阵:")
display(tf_df.round(4))

print("\n💡 TF分析：")
print(f"  - 每个文档的TF值总和 = 1.0")
print(f"  - 文档A TF总和: {sum(tfA.values()):.4f}")
print(f"  - 文档B TF总和: {sum(tfB.values()):.4f}")
print(f"  - 'cat' 在文档A中的TF: {tfA['cat']:.4f} (1/6)")
print(f"  - 'the' 在文档A中的TF: {tfA['the']:.4f} (1/6)")

## Step 8: 计算 IDF（逆文档频率 Inverse Document Frequency）


In [ ]:
print("="*80)
print("📉 Step 8: 计算逆文档频率 (IDF)")
print("="*80)


def computeIDF(wordDictList):
    """
    计算逆文档频率 (Inverse Document Frequency)

    公式: IDF(t) = log((N + 1) / (ni + 1))
        N: 文档总数
        ni: 包含词t的文档数

    Args:
        wordDictList: 多个文档的词频字典列表

    Returns:
        idfDict: 每个词的IDF值
    """
    # 初始化IDF字典
    idfDict = dict.fromkeys(wordDictList[0], 0)
    N = len(wordDictList)

    # 统计包含每个词的文档数（Ni）
    for wordDict in wordDictList:
        for word, count in wordDict.items():
            if count > 0:
                idfDict[word] += 1

    # 计算IDF
    for word, ni in idfDict.items():
        idfDict[word] = math.log10((N + 1) / (ni + 1))

    return idfDict


# 计算IDF
idfs = computeIDF([wordDictA, wordDictB])

# 展示结果
print("\nIDF值:")
idf_df = pd.DataFrame([idfs], index=['IDF值'])
display(idf_df.round(4))

print("\n💡 IDF分析：")
print(f"  - N = 2 (文档总数)")
print(f"  - 'the', 'sat', 'on', 'my' 出现在2个文档中，IDF = log(3/3) = 0")
print(
    f"  - 'cat', 'bed', 'dog', 'knees' 只在1个文档中，IDF = log(3/2) = {math.log10(3/2):.4f}")
print("\n  结论：常见词IDF低，稀有词IDF高")

## Step 9: 计算 TF-IDF


In [ ]:
print("="*80)
print("🎯 Step 9: 计算TF-IDF")
print("="*80)


def computeTFIDF(tf, idfs):
    """
    计算TF-IDF

    公式: TF-IDF(t,d) = TF(t,d) × IDF(t)

    Args:
        tf: 词频字典
        idfs: IDF字典

    Returns:
        tfidf: TF-IDF值字典
    """
    tfidf = {}
    for word, tfVal in tf.items():
        tfidf[word] = tfVal * idfs[word]
    return tfidf


# 计算两个文档的TF-IDF
tfidfA = computeTFIDF(tfA, idfs)
tfidfB = computeTFIDF(tfB, idfs)

# 展示结果
print("\nTF-IDF矩阵:")
tfidf_df = pd.DataFrame([tfidfA, tfidfB], index=['文档A TF-IDF', '文档B TF-IDF'])
display(tfidf_df.round(6))

print("\n💡 TF-IDF分析：")
print("  - 'the', 'sat', 'on', 'my': TF-IDF = 0 (常见词被抑制)")
print("  - 'cat', 'bed': 在文档A中有较高权重")
print("  - 'dog', 'knees': 在文档B中有较高权重")
print("\n✅ TF-IDF成功区分了文档特征！")

## Step 10: 手动实现总结


In [ ]:
# 10.1 完整对比表格
print("="*80)
print("📊 Step 10: TF-IDF完整对比")
print("="*80)

# 合并所有结果
comparison_df = pd.DataFrame({
    '词': list(wordSet),
    '文档A词频': [wordDictA[w] for w in wordSet],
    '文档B词频': [wordDictB[w] for w in wordSet],
    '文档A TF': [tfA[w] for w in wordSet],
    '文档B TF': [tfB[w] for w in wordSet],
    'IDF': [idfs[w] for w in wordSet],
    '文档A TF-IDF': [tfidfA[w] for w in wordSet],
    '文档B TF-IDF': [tfidfB[w] for w in wordSet]
})

# 排序：按IDF降序（稀有词在前）
comparison_df = comparison_df.sort_values('IDF', ascending=False)
print("\n完整TF-IDF计算结果:")
display(comparison_df.round(4))

# 10.2 关键发现
print("\n" + "="*80)
print("🔍 关键发现")
print("="*80)

print("\n1️⃣ 为什么'The/sat/on/my'的TF-IDF为0？")
print("   - 这些词在两个文档中都出现（ni=2, N=2）")
print("   - IDF = log((2+1)/(2+1)) = log(1) = 0")
print("   - 常见词对区分文档没有帮助，被TF-IDF抑制")

print("\n2️⃣ 为什么'cat/dog/bed/knees'的TF-IDF较高？")
print("   - 这些词只在1个文档中出现（ni=1, N=2）")
print(f"   - IDF = log((2+1)/(1+1)) = log(1.5) ≈ {math.log10(1.5):.4f}")
print("   - 特有词能够很好地区分文档内容")

print("\n3️⃣ TF-IDF的核心价值")
print("   ✅ 降低常见词权重（如the, a, is等停用词）")
print("   ✅ 提升特有词权重（能够代表文档特征的词汇）")
print("   ✅ 将文档转换为数值向量，便于计算相似度")

---

# 阶段三：使用 scikit-learn 实现 TF-IDF


## Step 11: 小样本验证


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
print("="*80)
print("🔬 Step 11: sklearn实现与手动实现对比")
print("="*80)

# 11.1 使用同样的示例数据
corpus = [docA.lower(), docB.lower()]
print(f"语料库: {corpus}")

# 11.2 使用TfidfVectorizer

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)

# 获取特征名
feature_names = vectorizer.get_feature_names_out()
print(f"\n特征词: {feature_names}")

# 转换为DataFrame
sklearn_tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=feature_names,
    index=['文档A (sklearn)', '文档B (sklearn)']
)
print("\nsklearn TF-IDF结果:")
display(sklearn_tfidf_df.round(6))

# 11.3 对比手动实现和sklearn实现
print("\n" + "="*80)
print("📊 两种实现对比")
print("="*80)

print("\n手动实现 (公式: TF × IDF):")
display(tfidf_df.round(6))

print("\nsklearn实现 (公式: TF × IDF + L2归一化):")
display(sklearn_tfidf_df.round(6))

print("\n💡 重要发现：")
print("  - sklearn默认进行了L2归一化（每行向量长度为1）")
print("  - 归一化后更适合计算余弦相似度")
print("  - 相对权重趋势一致，但绝对值不同")

## Step 12: 探索 TfidfVectorizer 参数


In [ ]:
print("="*80)
print("⚙️ Step 12: TfidfVectorizer参数探索")
print("="*80)

sample_texts = food_reviews_df['text'].head(5000).tolist()

# 12.1 max_features - 限制特征数量
print("\n1️⃣ max_features - 限制最大特征数")
print("-" * 60)

vectorizer_100 = TfidfVectorizer(max_features=100)
vectorizer_1000 = TfidfVectorizer(max_features=1000)

tfidf_100 = vectorizer_100.fit_transform(sample_texts)
tfidf_1000 = vectorizer_1000.fit_transform(sample_texts)

print(
    f"max_features=100: 矩阵形状 {tfidf_100.shape}, 特征示例: {vectorizer_100.get_feature_names_out()[:10]}")
print(
    f"max_features=1000: 矩阵形状 {tfidf_1000.shape}, 特征示例: {vectorizer_1000.get_feature_names_out()[:10]}")

# 12.2 min_df/max_df - 文档频率过滤
print("\n2️⃣ min_df/max_df - 文档频率过滤")
print("-" * 60)

# min_df: 忽略在少于min_df文档中出现的词
# max_df: 忽略在多于max_df文档中出现的词

vectorizer_filtered = TfidfVectorizer(
    min_df=1,      # 至少出现在1个文档中（默认值）
    max_df=0.8     # 忽略在80%以上文档中出现的词（停用词）
)

tfidf_filtered = vectorizer_filtered.fit_transform(sample_texts)
print(f"min_df=1, max_df=0.8: 矩阵形状 {tfidf_filtered.shape}")
print(f"特征数量: {len(vectorizer_filtered.get_feature_names_out())}")

In [ ]:
# 12.3 ngram_range - n-gram范围
print("\n3️⃣ ngram_range - n-gram范围")
print("-" * 60)

# unigrams (默认)
vec_unigram = TfidfVectorizer(ngram_range=(1, 1), max_features=50)
# unigrams + bigrams
vec_bigram = TfidfVectorizer(ngram_range=(1, 2), max_features=50)
# bigrams only
vec_bigram_only = TfidfVectorizer(ngram_range=(2, 2), max_features=50)

tfidf_unigram = vec_unigram.fit_transform(sample_texts)
tfidf_bigram = vec_bigram.fit_transform(sample_texts)
tfidf_bigram_only = vec_bigram_only.fit_transform(sample_texts)

print(f"unigram (1,1): 特征示例 {vec_unigram.get_feature_names_out()[:10]}")
print(f"unigram+bigram (1,2): 特征示例 {vec_bigram.get_feature_names_out()[:10]}")
print(
    f"bigram only (2,2): 特征示例 {vec_bigram_only.get_feature_names_out()[:10]}")

# 12.4 stop_words - 停用词处理
print("\n4️⃣ stop_words - 停用词处理")
print("-" * 60)

# 使用英语停用词
vec_with_stopwords = TfidfVectorizer(max_features=50)
vec_no_stopwords = TfidfVectorizer(stop_words='english', max_features=50)

tfidf_with = vec_with_stopwords.fit_transform(sample_texts)
tfidf_no = vec_no_stopwords.fit_transform(sample_texts)

print(f"包含停用词: {vec_with_stopwords.get_feature_names_out()[:15]}")
print(f"去除停用词: {vec_no_stopwords.get_feature_names_out()[:15]}")

print("\n💡 对比：去除停用词后，'the', 'and', 'to'等常见词被过滤")

## Step 13: 应用到 Yelp 评论数据


In [ ]:
print("="*80)
print("🍽️ Step 13: 对Yelp评论进行TF-IDF向量化")
print("="*80)

# 13.1 准备评论文本
print(f"\n准备数据: {len(food_reviews_df)} 条评论")

custom_stop_words = list(TfidfVectorizer(
    stop_words='english').get_stop_words())
# 泛情感词 - 每家餐厅评论里都会出现，没有区分度
sentiment_stops = ['delicious', 'love', 'amazing', 'excellent',
                   'friendly', 'definitely', 'favorite', 'perfect',
                   'horrible', 'terrible', 'worst', 'awful']
custom_stop_words += ['food', 'place', 'good', 'great', 'really',
                      'just', 'like', 'restaurant', 'service',
                      'time', 'nice', 've', 'ordered', 'best']
# 功能词 - 对推荐无意义
functional_stops = ['got', 'don', 'came', 'went', 'come', 'going',
                    'try', 'tried', 'little', 'wait', 'said',
                    'asked', 'told', 'took', 'make', 'know']

custom_stop_words += sentiment_stops + functional_stops

# 提高 min_df，降低 max_df
vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=5,          # 提高
    max_df=0.5,        # 降低，更激进地过滤高频词
    stop_words=custom_stop_words,
    token_pattern=r'(?u)\b[a-zA-Z]{2,}\b',  # 只保留纯字母、至少2个字符的词
    ngram_range=(1, 2),  # bigram 能捕捉 "pad thai", "fish taco" 等
    norm='l2'
)

print("\nVectorizer配置:")
print(f"  max_features: 5000")
print(f"  min_df: {vectorizer.min_df}")
print(f"  max_df: {vectorizer.max_df}")
print(f"  stop_words: 'english'")
print(f"  ngram_range: (1, 2)")

# 13.3 执行向量化
print("\n正在向量化...")
review_tfidf = vectorizer.fit_transform(food_reviews_df['text'])

print(f"\n✅ 向量化完成！")
print(f"TF-IDF矩阵形状: {review_tfidf.shape}")
print(f"  - 行数: {review_tfidf.shape[0]} (评论数)")
print(f"  - 列数: {review_tfidf.shape[1]} (特征词数)")
print(
    f"  - 稀疏度: {(1 - review_tfidf.nnz / (review_tfidf.shape[0] * review_tfidf.shape[1])) * 100:.2f}%")

In [ ]:
# 13.4 查看特征词表
print("\n" + "="*80)
print("📖 特征词表示例")
print("="*80)

feature_names = vectorizer.get_feature_names_out()
print(f"\n总特征数: {len(feature_names)}")
print(f"\n前50个特征词:")
print(feature_names[:50])

print(f"\n后50个特征词:")
print(feature_names[-50:])

# 13.5 查看特定关键词
food_keywords = ['delicious', 'tasty', 'spicy', 'sweet',
                 'fresh', 'good', 'great', 'amazing', 'terrible', 'bad']
found_keywords = [w for w in food_keywords if w in feature_names]
print(f"\n🍽️ 找到的美食相关词: {found_keywords}")

## Step 14: 特征词分析


In [ ]:
print("="*80)
print("📊 Step 14: 特征词权重分析")
print("="*80)

# 14.1 查看TF-IDF权重最高的词
print("\n1️⃣ 全局TF-IDF权重最高的词")
print("-" * 60)

# 计算每个特征的平均TF-IDF权重
mean_tfidf = np.array(review_tfidf.mean(axis=0)).flatten()
feature_weights = list(zip(feature_names, mean_tfidf))
feature_weights.sort(key=lambda x: x[1], reverse=True)

print("\nTop 20 平均TF-IDF权重最高的词:")
for word, weight in feature_weights[:20]:
    print(f"  {word}: {weight:.6f}")

# 14.2 可视化词云
print("\n2️⃣ 词云可视化")
print("-" * 60)

# 创建词云（使用前100个高频词）
wordcloud_dict = {word: weight for word, weight in feature_weights[:100]}

wordcloud = WordCloud(
    width=800, height=400,
    background_color='white',
    colormap='viridis',
    max_words=100
).generate_from_frequencies(wordcloud_dict)

plt.figure(figsize=(12, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Top TF-IDF Features Word Cloud')
plt.tight_layout()
plt.show()

In [ ]:
# 14.3 查看单条评论的特征表示
print("\n3️⃣ 单条评论的TF-IDF特征表示")
print("-" * 60)

# 选择一条评论
sample_idx = 0
sample_review = food_reviews_df.iloc[sample_idx]
sample_biz_name = food_business_df[food_business_df['business_id']
                                   == sample_review['business_id']]['name'].values
sample_biz_name = sample_biz_name[0] if len(sample_biz_name) > 0 else 'Unknown'

print(f"\n商家: {sample_biz_name}")
print(f"评分: {sample_review['stars']} stars")
print(f"评论: {sample_review['text'][:200]}...")

# 获取该评论的TF-IDF向量
sample_vector = review_tfidf[sample_idx]

# 获取非零特征的索引和值
nonzero_indices = sample_vector.nonzero()[1]
nonzero_values = sample_vector.data

# 排序并显示Top特征
feature_value_pairs = [(feature_names[i], v)
                       for i, v in zip(nonzero_indices, nonzero_values)]
feature_value_pairs.sort(key=lambda x: x[1], reverse=True)

print(f"\n该评论的Top 20 TF-IDF特征:")
for word, value in feature_value_pairs[:20]:
    print(f"  {word}: {value:.4f}")

---

# 阶段四：构建餐厅特征向量


## Step 15: 按餐厅聚合评论


In [ ]:
print("="*80)
print("🍴 Step 15: 按餐厅聚合评论文本")
print("="*80)

# 15.1 聚合评论文本
print("\n1️⃣ 聚合每个商家的所有评论")
print("-" * 60)

# 按business_id分组，合并所有评论文本
business_reviews = food_reviews_df.groupby('business_id').agg({
    'text': lambda x: ' '.join(x),  # 合并所有评论
    'stars': 'mean',                 # 平均评分
    'review_id': 'count'             # 评论数
}).reset_index()

business_reviews.columns = ['business_id',
                            'aggregated_text', 'avg_stars', 'review_count']

# 合并商家信息
business_profiles = pd.merge(
    food_business_df[['business_id', 'name', 'categories', 'stars']],
    business_reviews,
    on='business_id',
    how='inner'
)

# business_profiles = business_profiles[business_profiles['review_count'] >= 3]
print(f"\n商家画像数据: {len(business_profiles)} 家")
print(f"\n数据预览:")
display(business_profiles[['name', 'categories',
        'avg_stars', 'review_count']].head())

print(f"\n统计信息:")
print(business_profiles['review_count'].describe())
# 过滤评论太少的商家

In [ ]:
# 15.2 将categories添加到文本特征中
print("\n2️⃣ 融合Categories作为补充特征")
print("-" * 60)


def preprocess_categories(categories):
    if pd.isna(categories):
        return ""
    cats = []
    for c in categories.split(','):
        c = c.strip().lower()
        c = re.sub(r'[^a-z0-9\s]', '', c)  # 去掉特殊字符
        c = re.sub(r'\s+', '_', c.strip())   # 多个空格合并后转下划线
        if c and c != '_':
            cats.append(c)
    return ' '.join(cats)

# "Health & Medical" → "health_medical"✅
# "Ice Cream & Frozen Yogurt" → "ice_cream_frozen_yogurt" ✅

# 验证效果:
# "American (Traditional)" → "american_traditional"
# "Breakfast & Brunch"     → "breakfast_brunch"  
# "Ice Cream & Frozen Yogurt" → "ice_cream_frozen_yogurt"

# 将categories拼接到文本中（可配置权重）
business_profiles['categories_text'] = business_profiles['categories'].apply(
    preprocess_categories)

# 组合文本：评论 + categories
CATEGORY_WEIGHT = 5  # 重复5次增加权重

business_profiles['combined_text'] = (
    business_profiles['aggregated_text'] + ' ' +
    (business_profiles['categories_text'] + ' ') * CATEGORY_WEIGHT
)
# 查看组合效果
print("\n组合文本示例:")
sample_biz = business_profiles.iloc[0]
print(f"\n商家: {sample_biz['name']}")
print(f"\n原始评论长度: {len(sample_biz['aggregated_text'])} 字符")
print(f"Categories: {sample_biz['categories_text'][:100]}...")
print(f"\n组合后文本长度: {len(sample_biz['combined_text'])} 字符")

## Step 16: 构建餐厅 TF-IDF 特征


In [ ]:
print("=" * 80)
print("🍴 Step 16: 构建餐厅级别TF-IDF 特征")
print("=" * 80)

# 16.1 配置 TF-IDF 向量化器（餐厅级别）
print("\n1️⃣ 配置向量化器")
print("-" * 60)

# 扩充停用词
sentiment_stops = ['delicious', 'love', 'amazing', 'excellent',
                   'friendly', 'definitely', 'favorite', 'perfect',
                   'horrible', 'terrible', 'worst', 'awful']

functional_stops = ['got', 'don', 'came', 'went', 'come', 'going',
                    'try', 'tried', 'little', 'wait', 'said',
                    'asked', 'told', 'took', 'make', 'know']

business_stop_words = list(TfidfVectorizer(stop_words='english').get_stop_words())
business_stop_words += ['food', 'place', 'good', 'great', 'really',
                        'just', 'like', 'restaurant', 'service',
                        'time', 'nice', 've', 'ordered', 'best']
business_stop_words += sentiment_stops + functional_stops

business_vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=3,                # 至少出现在3家餐厅中
    max_df=0.5,                                  # 最多出现在50%的餐厅中
    stop_words=business_stop_words,
    ngram_range=(1, 2),
    token_pattern=r'(?u)\b[a-zA-Z_]{2,}\b',# 保留字母和下划线（支持 bubble_tea 等）
    norm='l2'
)

# 16.2 执行向量化
print("\n2️⃣ 执行向量化")
print("-" * 60)

business_tfidf = business_vectorizer.fit_transform(business_profiles['combined_text'])
business_feature_names = business_vectorizer.get_feature_names_out()

print(f"\n✅ 餐厅级别 TF-IDF 构建完成！")
print(f"  矩阵形状: {business_tfidf.shape}")
print(f"  -餐厅数: {business_tfidf.shape[0]}")
print(f"  - 特征词数: {business_tfidf.shape[1]}")
print(f"  - 稀疏度: {(1 - business_tfidf.nnz / (business_tfidf.shape[0] * business_tfidf.shape[1])) * 100:.2f}%")
print(f"  - 非零元素: {business_tfidf.nnz:,}")

# 16.3 查看 Top 特征词
print("\n3️⃣ 全局Top 30 特征词（按平均 TF-IDF 权重）")
print("-" * 60)

mean_tfidf_business = np.array(business_tfidf.mean(axis=0)).flatten()
top_indices = mean_tfidf_business.argsort()[::-1][:30]

for rank, idx in enumerate(top_indices, 1):
    print(f"  {rank:2d}. {business_feature_names[idx]:<25s} {mean_tfidf_business[idx]:.6f}")

# 16.4 检查 categories 特征是否被捕获
print("\n4️⃣ Categories 特征检查")
print("-" * 60)

category_features = [f for f in business_feature_names if '_' in f]
print(f"  含下划线的特征（来自categories）: {len(category_features)} 个")
if category_features:
    print(f"  示例: {category_features[:20]}")

## Step 17: 特征归一化


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
print("="*80)
print("📐 Step 17: 特征归一化")
print("="*80)

# 17.1 检查当前向量长度
print("\n1️⃣ 检查向量L2范数")
print("-" * 60)

# 计算L2范数（处理稀疏矩阵）
sample_norms = np.sqrt(business_tfidf[:5].power(2).sum(axis=1)).A1

print(f"\n前5家餐厅向量的L2范数:")
for i, norm in enumerate(sample_norms):
    print(f"  餐厅{i+1}: {norm:.6f}")  # 去掉 [0]

print(f"\n💡 所有向量L2范数都接近1.0，说明已经归一化")
print(f"   归一化后的向量适合直接计算余弦相似度")

# 17.2 可选：手动归一化（如果未归一化）
# from sklearn.preprocessing import normalize
# business_tfidf_normalized = normalize(business_tfidf, norm='l2', copy=False)

print("\n2️⃣ 验证归一化效果")
print("-" * 60)

# 计算两个餐厅向量的点积（等于余弦相似度）

cos_sim_01 = cosine_similarity(business_tfidf[0:1], business_tfidf[1:2])[0][0]
print(f"\n餐厅1和餐厅2的余弦相似度: {cos_sim_01:.4f}")

# 自己与自己应该为1
cos_sim_self = cosine_similarity(
    business_tfidf[0:1], business_tfidf[0:1])[0][0]
print(f"餐厅1与自己的余弦相似度: {cos_sim_self:.4f}")

---

# 阶段五：构建用户画像


## Step 18: 选择目标用户


In [ ]:
print("="*80)
print("👤 Step 18: 选择目标用户")
print("="*80)

# 18.1 统计用户评论数
print("\n1️⃣ 用户评论数统计")
print("-" * 60)

user_review_counts = food_reviews_df.groupby('user_id').agg({
    'review_id': 'count',
    'stars': ['mean', 'std'],
    'business_id': 'nunique'
}).reset_index()

user_review_counts.columns = [
    'user_id', 'review_count', 'avg_rating', 'rating_std', 'unique_businesses']
user_review_counts = user_review_counts.sort_values(
    'review_count', ascending=False)

print(f"\n用户统计:")
print(f"  总用户数: {len(user_review_counts)}")
print(user_review_counts['review_count'].describe())

# 18.2 选择示例用户
print("\n2️⃣ 选择示例用户")
print("-" * 60)

# 选择评论数适中（5-20条）的用户作为示例
candidate_users = user_review_counts[
    (user_review_counts['review_count'] >= 5) &
    (user_review_counts['review_count'] <= 20)
]

if len(candidate_users) > 0:
    sample_user_id = candidate_users.iloc[len(candidate_users)//2]['user_id']
else:
    sample_user_id = user_review_counts.iloc[10]['user_id']

print(f"\n选择的示例用户ID: {sample_user_id}")

# 获取该用户的所有评论
user_reviews = food_reviews_df[food_reviews_df['user_id'] == sample_user_id]
print(f"该用户评论数: {len(user_reviews)}")
print(f"平均评分: {user_reviews['stars'].mean():.2f}")

# 检查用户评过的餐厅有多少在 business_profiles 中
user_biz_ids = set(user_reviews['business_id'])
matched = user_biz_ids & set(business_profiles['business_id'])
print(f"用户评过 {len(user_biz_ids)} 家, 在候选池中: {len(matched)} 家")

# 查看用户历史
print("\n用户历史评论:")
for idx, row in user_reviews.iterrows():
    biz_name = food_business_df[food_business_df['business_id']
                                == row['business_id']]['name'].values
    biz_name = biz_name[0] if len(biz_name) > 0 else 'Unknown'
    print(f"  {biz_name} ({row['stars']}★): {row['text'][:80]}...")

## Step 19: 构建用户特征向量


In [ ]:
print("=" * 80)
print("🎭 Step 19: 构建用户特征向量（用户画像）")
print("=" * 80)

# 19.1 方法1：基于用户评过的餐厅（推荐）
print("\n1️⃣ 方法1：基于用户评过的餐厅TF-IDF向量加权平均")
print("-" * 60)

# 找到用户评过的餐厅在 business_profiles 中的位置
user_biz_ids = user_reviews['business_id'].values
matched_mask = business_profiles['business_id'].isin(user_biz_ids)
matched_indices = business_profiles[matched_mask].index.tolist()
# 转换为 business_tfidf 矩阵中的行索引
matched_positions = [business_profiles.index.get_loc(idx) for idx in matched_indices]

print(f"用户评过 {len(user_biz_ids)} 家, 在候选池中匹配: {len(matched_positions)} 家")

if len(matched_positions) > 0:
    # 获取匹配餐厅的 TF-IDF 向量
    user_biz_vectors = business_tfidf[matched_positions]
    
    # 获取对应评分作为权重
    matched_biz_ids = business_profiles.iloc[matched_positions]['business_id'].values
    user_stars = []
    for bid in matched_biz_ids:
        star = user_reviews[user_reviews['business_id'] == bid]['stars'].values[0]
        user_stars.append(star)
    user_stars = np.array(user_stars)
    
    # 归一化评分作为权重
    weights = (user_stars - 1) / 4
    weights = weights / weights.sum()
    
    # 加权平均
    user_biz_vectors_dense = user_biz_vectors.toarray()
    user_profile = np.average(user_biz_vectors_dense, axis=0, weights=weights)
    
    # L2 归一化
    user_profile = user_profile / np.linalg.norm(user_profile)
    
    print(f"用户画像向量维度: {user_profile.shape}")
    print(f"非零元素数: {np.count_nonzero(user_profile)}")
    print(f"权重分布: {dict(zip(matched_biz_ids[:5], weights[:5]))}")
    print(f"L2归一化后范数: {np.linalg.norm(user_profile):.6f}")

    # 查看用户画像的 Top 特征词
    print("\n📊 用户画像 Top 15 特征词:")
    top_indices = user_profile.argsort()[::-1][:15]
    for rank, idx in enumerate(top_indices, 1):
        print(f"  {rank:2d}. {business_feature_names[idx]:<25s} {user_profile[idx]:.6f}")

**简单平均与加权平均核心区别：**

| | 简单平均 | 加权平均 |
|---|---------|---------|
| **权重** | 每家餐厅 1/n | 按评分分配 |
| **语义** | 用户去过哪些 | 用户喜欢哪些 |
| **问题** | 低分餐厅也会影响画像 | 需要足够的评分方差 |
| **适用** | 评分缺失时的降级方案 | 有评分数据时的首选 |

In [ ]:
print("=" * 80)
print("⚖️ 简单平均 vs 加权平均 用户画像对比")
print("=" * 80)

# ============================================================
# 1. 简单平均：所有评过的餐厅一视同仁
# ============================================================
print("\n1️⃣ 简单平均（Simple Average）")
print("-" * 60)
print("公式: user_profile = mean(v1, v2, ..., vn)")
print("思路: 用户评过的每家餐厅权重相同，不管评了几星\n")

simple_profile = np.array(user_biz_vectors.mean(axis=0)).flatten()
simple_profile = simple_profile / np.linalg.norm(simple_profile)

print("Top 10 特征词:")
top_simple = simple_profile.argsort()[::-1][:10]
for rank, idx in enumerate(top_simple, 1):
    print(f"  {rank:2d}. {business_feature_names[idx]:<30s} {simple_profile[idx]:.6f}")

# ============================================================
# 2. 加权平均：评分越高的餐厅影响越大
# ============================================================
print(f"\n2️⃣ 加权平均（Rating-Weighted Average）")
print("-" * 60)
print("公式: user_profile = Σ(wi * vi) / Σ(wi)")
print("思路: 用户给5星的餐厅比给2星的更能代表其偏好\n")

# 展示每家餐厅的评分和权重
print("各餐厅权重分配:")
print(f"  {'餐厅':<30s} {'评分':>5s} {'归一化权重':>10s}")
print(f"  {'-'*50}")
for i, bid in enumerate(matched_biz_ids):
    biz_name = business_profiles[business_profiles['business_id'] == bid]['name'].values[0]
    raw_weight = (user_stars[i] - 1) / 4  # 1星→0, 5星→1
    print(f"  {biz_name[:30]:<30s} {user_stars[i]:>5.1f} {weights[i]:>10.4f}")

print("\nTop 10 特征词:")
top_weighted = user_profile.argsort()[::-1][:10]
for rank, idx in enumerate(top_weighted, 1):
    print(f"  {rank:2d}. {business_feature_names[idx]:<30s} {user_profile[idx]:.6f}")

# ============================================================
# 3. 对比分析
# ============================================================
print(f"\n3️⃣ 差异对比")
print("-" * 60)

# 计算两种画像的相似度
similarity = cosine_similarity(
    simple_profile.reshape(1, -1),
    user_profile.reshape(1, -1)
)[0][0]
print(f"\n两种画像的余弦相似度: {similarity:.4f}")
print(f"  → {'差异较小，评分比较均匀' if similarity > 0.95 else '差异明显，评分有明显偏好'}")

# 找出差异最大的特征词
diff = user_profile - simple_profile
top_diff_pos = diff.argsort()[::-1][:5]
top_diff_neg = diff.argsort()[:5]

print(f"\n加权后权重↑上升最多的词（用户高分餐厅的特征）:")
for idx in top_diff_pos:
    print(f"  📈 {business_feature_names[idx]:<30s} {simple_profile[idx]:.4f} → {user_profile[idx]:.4f} ({diff[idx]:+.4f})")

print(f"\n加权后权重↓下降最多的词（用户低分餐厅的特征）:")
for idx in top_diff_neg:
    print(f"  📉 {business_feature_names[idx]:<30s} {simple_profile[idx]:.4f} → {user_profile[idx]:.4f} ({diff[idx]:+.4f})")

print(f"\n💡 结论:")
print(f"  简单平均: 反映用户'去过什么餐厅'")
print(f"  加权平均: 反映用户'真正喜欢什么餐厅'")
print(f"  加权平均能更好地捕捉用户偏好，推荐更精准")

## Step 20: 用户-餐厅相似度计算


In [ ]:
from scipy.sparse import csr_matrix
print("="*80)
print("📏 Step 20: 计算用户-餐厅相似度")
print("="*80)

# 20.1 计算余弦相似度
print("\n1️⃣ 计算用户与所有餐厅的余弦相似度")
print("-" * 60)

# 将用户画像转换为稀疏矩阵格式
user_profile_sparse = csr_matrix(user_profile.reshape(1, -1))

# 使用之前定义的餐厅向量化器来转换用户评论
# 注意：需要将用户评论文本用business_vectorizer转换

# 聚合用户评论
user_all_text = ' '.join(user_reviews['text'].tolist())
user_biz_vector = business_vectorizer.transform([user_all_text])

# 计算与所有餐厅的相似度
similarities = cosine_similarity(user_biz_vector, business_tfidf).flatten()

print(f"\n相似度计算完成！")
print(f"相似度范围: [{similarities.min():.4f}, {similarities.max():.4f}]")
print(f"平均相似度: {similarities.mean():.4f}")

# 方法A：用之前构建的用户画像（加权餐厅向量）
similarities_A = cosine_similarity(
    user_profile.reshape(1, -1), business_tfidf
).flatten()

# 方法B：当前用的（用户评论直接transform）
similarities_B = similarities  # 当前结果

print(f"方法A 相似度范围: [{similarities_A.min():.4f}, {similarities_A.max():.4f}]")
print(f"方法B 相似度范围: [{similarities_B.min():.4f}, {similarities_B.max():.4f}]")

# 20.2 查看最相似的餐厅
print("\n2️⃣ 查看相似度Top 10餐厅")
print("-" * 60)

# 获取相似度排序索引
similarity_idx = similarities_A.argsort()[::-1]

# 排除用户已评过的餐厅
user_visited = set(user_reviews['business_id'])
for i in range(len(similarities)):
    if business_profiles.iloc[i]['business_id'] in user_visited:
        similarities[i] = -1  # 排除

print("\n与用户最相似的10家餐厅:")
for i in range(10):
    idx = similarity_idx[i]
    biz = business_profiles.iloc[idx]
    print(
        f"  {i+1}. {biz['name']} ({biz['stars']}★) - 相似度: {similarities_A[idx]:.4f}")

---

# 阶段六：基于内容的推荐


## Step 21: 实现推荐函数


In [ ]:
def content_based_recommendations(
    user_id,
    business_tfidf,
    business_profiles,
    business_vectorizer,
    food_reviews_df,
    top_k=10
):
    """
    基于内容的餐厅推荐
    
    Args:
        user_id: 目标用户ID
        business_tfidf: 餐厅TF-IDF矩阵 (sparse matrix)
        business_profiles: 餐厅画像DataFrame
        business_vectorizer: 训练好的TfidfVectorizer
        food_reviews_df: 食品评论DataFrame
        top_k: 推荐数量
    
    Returns:
        recommendations: 推荐结果DataFrameuser_profile: 用户画像向量
    """
    
    # ========================================
    # 1. 获取用户历史评论
    # ========================================
    user_reviews = food_reviews_df[food_reviews_df['user_id'] == user_id]
    
    if len(user_reviews) == 0:
        print(f"⚠️ 用户 {user_id} 没有评论记录，无法生成推荐")
        return None, None
    
    # ========================================
    # 2. 构建用户画像向量（加权平均法）
    # ========================================
    user_biz_ids = set(user_reviews['business_id'])
    matched_mask = business_profiles['business_id'].isin(user_biz_ids)
    matched_positions = [
        business_profiles.index.get_loc(idx)
        for idx in business_profiles[matched_mask].index.tolist()
    ]
    
    if len(matched_positions) == 0:
        # 降级方案：用用户评论文本直接transform
        print("⚠️ 用户评过的餐厅不在候选池中，使用评论文本降级方案")
        user_all_text = ' '.join(user_reviews['text'].tolist())
        user_profile = business_vectorizer.transform([user_all_text]).toarray().flatten()
        user_profile = user_profile / (np.linalg.norm(user_profile) + 1e-10)
    else:
        # 主方案：基于评过的餐厅TF-IDF向量加权平均
        matched_biz_ids = business_profiles.iloc[matched_positions]['business_id'].values
        user_vectors = business_tfidf[matched_positions].toarray()
        # 用评分作为权重（评分越高，该餐厅对画像贡献越大）
        weights = []
        for bid in matched_biz_ids:
            star = user_reviews[user_reviews['business_id'] == bid]['stars'].values[0]
            weights.append(star)
        weights = np.array(weights)
        
        # 归一化权重：(star -1) / 4 → [0, 1]，避免1星餐厅权重为负
        weights = (weights - 1) / 4
        # 防止全零权重
        if weights.sum() == 0:
            weights = np.ones(len(weights))
        weights = weights / weights.sum()
        
        # 加权平均 + L2归一化
        user_profile = np.average(user_vectors, axis=0, weights=weights)
        user_profile = user_profile / (np.linalg.norm(user_profile) + 1e-10)
    
    # ========================================
    # 3. 计算余弦相似度
    # ========================================
    similarities = cosine_similarity(
        user_profile.reshape(1, -1), business_tfidf
    ).flatten()
    
    # ========================================
    # 4. 过滤 & 排序
    # ========================================
    results = business_profiles.copy()
    results['similarity'] = similarities
    
    # 排除用户已评过的餐厅
    results = results[~results['business_id'].isin(user_biz_ids)]
    
    # 排除低评分餐厅（< 3.0★）
    results = results[results['stars'] >= 3.0]
    
    # 排除评论数过少的餐厅（可靠性低）
    results = results[results['review_count'] >= 3]
    
    # 按相似度降序排序
    results = results.sort_values('similarity', ascending=False)
    
    # ========================================
    # 5. 输出推荐结果
    # ========================================
    recommendations = results.head(top_k).copy()
    recommendations = recommendations[[
        'business_id', 'name', 'categories', 'stars',
        'avg_stars', 'review_count', 'similarity'
    ]].reset_index(drop=True)
    recommendations.index = range(1, len(recommendations) + 1)
    recommendations.index.name = 'rank'
    
    #========================================
    # 6. 打印推荐报告
    # ========================================
    print("=" * 80)
    print(f"🍽️ 为用户 {user_id[:8]}... 的推荐结果")
    print("=" * 80)
    #用户历史摘要
    print(f"\n📋 用户历史: {len(user_reviews)} 条评论, "
          f"匹配候选池: {len(matched_positions)} 家, "
          f"平均评分: {user_reviews['stars'].mean():.1f}★")
    
    # 用户画像Top特征
    top_feature_idx = user_profile.argsort()[::-1][:10]
    feature_names = business_vectorizer.get_feature_names_out()
    top_features = [feature_names[i] for i in top_feature_idx]
    print(f"🎭 用户画像关键词: {', '.join(top_features[:5])}")
    
    # 推荐列表
    print(f"\n🏆 Top {top_k} 推荐:")
    print("-" * 70)
    for rank, row in recommendations.iterrows():
        print(f"{rank:2d}. {row['name']:<35s} {row['stars']}★  "
              f"({row['review_count']:3d}条评论)  "
              f"相似度: {row['similarity']:.4f}")
        print(f"      📂 {row['categories'][:70]}")
    
    return recommendations, user_profile


# ========================================
# 测试推荐函数
# ========================================
recommendations, user_profile = content_based_recommendations(
    user_id=sample_user_id,
    business_tfidf=business_tfidf,
    business_profiles=business_profiles,
    business_vectorizer=business_vectorizer,
    food_reviews_df=food_reviews_df,
    top_k=10
)

---

# 阶段七：评估与可视化


## Step 25: 推荐结果可视化


In [ ]:
print("=" * 80)
print("📊 Step 21: 推荐结果可视化")
print("=" * 80)

fig = plt.figure(figsize=(20, 16))

# ========================================
# 1. 用户画像词云
# ========================================
ax1 = fig.add_subplot(2, 2, 1)

feature_names = business_vectorizer.get_feature_names_out()
# 构建词频字典（TF-IDF权重作为词频）
word_weights = {
    feature_names[i]: float(user_profile[i])
    for i in range(len(feature_names))
    if user_profile[i] > 0
}

wordcloud = WordCloud(
    width=600, height=400,
    background_color='white',
    colormap='viridis',
    max_words=80,
    prefer_horizontal=0.7
).generate_from_frequencies(word_weights)

ax1.imshow(wordcloud, interpolation='bilinear')
ax1.set_title('🎭 用户画像词云', fontsize=14, fontweight='bold')
ax1.axis('off')

# ========================================
# 2. Top 10 推荐餐厅相似度条形图
# ========================================
ax2 = fig.add_subplot(2, 2, 2)

rec_names = [name[:25] + '...' if len(name) > 25 else name 
             for name in recommendations['name']]
rec_similarities = recommendations['similarity'].values
rec_stars = recommendations['stars'].values

# 根据评分着色
colors = plt.cm.RdYlGn((rec_stars - 2.5) / 2.5)  # 3.0-5.0 → 0.2-1.0

bars = ax2.barh(range(len(rec_names) - 1, -1, -1), rec_similarities, color=colors, edgecolor='gray', linewidth=0.5)

ax2.set_yticks(range(len(rec_names) - 1, -1, -1))
ax2.set_yticklabels(rec_names, fontsize=10)
ax2.set_xlabel('余弦相似度', fontsize=12)
ax2.set_title('🏆 Top 10 推荐餐厅 (颜色=评分)', fontsize=14, fontweight='bold')

# 在条形上标注相似度和评分
for i, (sim, star) in enumerate(zip(rec_similarities, rec_stars)):
    ax2.text(sim + 0.003, len(rec_names) - 1 - i, f'{sim:.3f} | {star}★',
             va='center', fontsize=9)

# 添加颜色图例
sm = plt.cm.ScalarMappable(cmap='RdYlGn', norm=plt.Normalize(vmin=3.0, vmax=5.0))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax2, shrink=0.6, pad=0.02)
cbar.set_label('餐厅评分', fontsize=10)

# ========================================
# 3. 推荐餐厅类别分布（饼图）
# ========================================
ax3 = fig.add_subplot(2, 2, 3)

# 提取推荐餐厅的所有类别
all_categories = []
for cats in recommendations['categories']:
    if pd.notna(cats):
        for c in cats.split(','):
            c = c.strip()
            if c not in ['Restaurants', 'Food']:  # 排除泛类
                all_categories.append(c)

cat_counts = Counter(all_categories)
top_cats = cat_counts.most_common(8)
labels = [c[0] for c in top_cats]
sizes = [c[1] for c in top_cats]

# 其他类别合并
other_count = sum(cat_counts.values()) - sum(sizes)
if other_count > 0:
    labels.append('Others')
    sizes.append(other_count)

wedges, texts, autotexts = ax3.pie(
    sizes, labels=labels, autopct='%1.0f%%',
    startangle=90, pctdistance=0.85,
    colors=sns.color_palette('Set2', len(labels))
)
plt.setp(texts, fontsize=9)
plt.setp(autotexts, fontsize=8)
ax3.set_title('📂 推荐餐厅类别分布', fontsize=14, fontweight='bold')

# ========================================
# 4. 用户历史 vs 推荐 对比雷达图
# ========================================
ax4 = fig.add_subplot(2, 2, 4, polar=True)

# 选取关键维度对比
dimensions = ['mexican', 'american_new', 'asian_fusion', 'seafood',
              'breakfast_brunch', 'italian', 'bars', 'bakeries']
dim_labels = ['Mexican', 'American New', 'Asian Fusion', 'Seafood',
              'Brunch', 'Italian', 'Bars', 'Bakeries']

# 用户画像在各维度的权重
user_dim_values = []
for dim in dimensions:
    if dim in feature_names:
        idx = np.where(feature_names == dim)[0][0]
        user_dim_values.append(user_profile[idx])
    else:
        user_dim_values.append(0)

# 推荐餐厅在各维度的平均权重
rec_positions = [
    business_profiles.index.get_loc(
        business_profiles[business_profiles['business_id'] == bid].index[0]
    )
    for bid in recommendations['business_id']
    if bid in business_profiles['business_id'].values
]
rec_vectors = business_tfidf[rec_positions].toarray()
rec_mean = rec_vectors.mean(axis=0)

rec_dim_values = []
for dim in dimensions:
    if dim in feature_names:
        idx = np.where(feature_names == dim)[0][0]
        rec_dim_values.append(rec_mean[idx])
    else:
        rec_dim_values.append(0)

# 归一化到 [0, 1]
max_val = max(max(user_dim_values), max(rec_dim_values), 1e-10)
user_dim_values = [v / max_val for v in user_dim_values]
rec_dim_values = [v / max_val for v in rec_dim_values]

# 绘制雷达图
angles = np.linspace(0, 2 * np.pi, len(dimensions), endpoint=False).tolist()
angles += angles[:1]
user_dim_values += user_dim_values[:1]
rec_dim_values += rec_dim_values[:1]

ax4.plot(angles, user_dim_values, 'o-', linewidth=2, label='用户画像', color='#2196F3')
ax4.fill(angles, user_dim_values, alpha=0.15, color='#2196F3')
ax4.plot(angles, rec_dim_values, 'o-', linewidth=2, label='推荐结果', color='#FF5722')
ax4.fill(angles, rec_dim_values, alpha=0.15, color='#FF5722')

ax4.set_xticks(angles[:-1])
ax4.set_xticklabels(dim_labels, fontsize=9)
ax4.set_title('🎯 用户画像 vs 推荐匹配度', fontsize=14, fontweight='bold', pad=20)
ax4.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)

plt.tight_layout(pad=3.0)
plt.savefig('recommendation_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ 可视化已保存为 recommendation_visualization.png")

## Step 26: 简单评估


In [ ]:
print("=" * 80)
print("📊 Step 21: 推荐效果评估")
print("=" * 80)

# ========================================
# 21.1 准备评估：多用户测试
# ========================================
print("\n1️⃣ 多用户批量推荐")
print("-" * 60)

# 选择有足够评论的用户
eval_candidates = user_review_counts[
    (user_review_counts['review_count'] >= 5) &
    (user_review_counts['review_count'] <= 30)
]

# 随机选择50个用户评估
np.random.seed(42)
eval_user_ids = eval_candidates.sample(min(50, len(eval_candidates)))['user_id'].values
print(f"评估用户数: {len(eval_user_ids)}")

# 批量生成推荐
all_results = []
valid_users = 0

for uid in eval_user_ids:
    try:
        recs, profile = content_based_recommendations(
            user_id=uid,
            business_tfidf=business_tfidf,
            business_profiles=business_profiles,
            business_vectorizer=business_vectorizer,
            food_reviews_df=food_reviews_df,
            top_k=10
        )
        if recs is not None and len(recs) > 0:
            recs['user_id'] = uid
            all_results.append(recs)
            valid_users += 1
    except Exception as e:
        continue

# 静默运行，汇总结果
all_recs_df = pd.concat(all_results, ignore_index=True)
print(f"成功生成推荐的用户: {valid_users} / {len(eval_user_ids)}")
print(f"总推荐条目: {len(all_recs_df)}")

# ========================================
# 21.2 指标1：推荐餐厅的平均评分
# ========================================
print("\n" + "=" * 80)
print("⭐ 指标1: 推荐餐厅评分质量")
print("=" * 80)

rec_avg_stars = all_recs_df['stars'].mean()
rec_avg_similarity = all_recs_df['similarity'].mean()

# 候选池整体评分
pool_avg_stars = business_profiles['stars'].mean()

print(f"\n  推荐餐厅平均评分:   {rec_avg_stars:.4f} ★")
print(f"  候选池整体平均评分: {pool_avg_stars:.4f} ★")
print(f"  评分提升:           +{rec_avg_stars - pool_avg_stars:.4f} ★")
print(f"  平均相似度:         {rec_avg_similarity:.4f}")

# 推荐评分分布
print(f"\n  推荐评分分布:")
star_dist = all_recs_df['stars'].value_counts().sort_index()
for star, count in star_dist.items():
    pct = count / len(all_recs_df) * 100
    bar = '█' * int(pct / 2)
    print(f"    {star}★: {count:4d} ({pct:5.1f}%) {bar}")

# ========================================
# 21.3指标2：与随机推荐对比
# ========================================
print("\n" + "=" * 80)
print("🎲 指标2: 与随机推荐对比")
print("=" * 80)

# 过滤条件与推荐函数一致（stars >= 3.0, review_count >= 3）
eligible_pool = business_profiles[
    (business_profiles['stars'] >= 3.0) &
    (business_profiles['review_count'] >= 3)
]

n_random_trials = 1000
random_avg_stars_list = []

np.random.seed(42)
for _ in range(n_random_trials):
    random_recs = eligible_pool.sample(10)
    random_avg_stars_list.append(random_recs['stars'].mean())

random_avg_stars = np.mean(random_avg_stars_list)
random_std_stars = np.std(random_avg_stars_list)

print(f"\n{'指标':<20s} {'内容推荐':>12s} {'随机推荐':>12s} {'提升':>10s}")
print(f"  {'-'*58}")
print(f"  {'平均评分 (★)':<20s} {rec_avg_stars:>12.4f} {random_avg_stars:>12.4f} {rec_avg_stars - random_avg_stars:>+10.4f}")
print(f"  {'平均相似度':<20s} {rec_avg_similarity:>12.4f} {'N/A':>12s} {'':>10s}")

# 统计显著性（t检验）
from scipy import stats

# 每个用户的推荐平均评分
per_user_avg_stars = [r['stars'].mean() for r in all_results]

t_stat, p_value = stats.ttest_1samp(per_user_avg_stars, random_avg_stars)
print(f"\n  统计检验: t={t_stat:.4f}, p={p_value:.4f}", end="")
print(f"  {'✅ 显著优于随机 (p<0.05)' if p_value < 0.05 else '❌ 无显著差异'}")

# 分布对比可视化
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.hist(random_avg_stars_list, bins=30, alpha=0.6, label=f'随机推荐 (μ={random_avg_stars:.2f})', color='gray')
ax.axvline(rec_avg_stars, color='red', linewidth=2, linestyle='--', 
           label=f'内容推荐 (μ={rec_avg_stars:.2f})')
ax.set_xlabel('平均评分 (★)')
ax.set_ylabel('频次')
ax.set_title('内容推荐 vs 随机推荐 评分分布对比')
ax.legend()
plt.tight_layout()
plt.show()

# ========================================
# 21.4 指标3：推荐多样性
# ========================================
print("\n" + "=" * 80)
print("🌈 指标3: 推荐多样性")
print("=" * 80)

# 3a. 类别覆盖率
all_rec_categories = []
for cats in all_recs_df['categories'].dropna():
    for c in cats.split(','):
        c = c.strip()
        if c and c not in ['Restaurants', 'Food']:
            all_rec_categories.append(c)

unique_rec_cats = set(all_rec_categories)
total_cats_in_pool = set()
for cats in business_profiles['categories'].dropna():
    for c in cats.split(','):
        c = c.strip()
        if c and c not in ['Restaurants', 'Food']:
            total_cats_in_pool.add(c)

category_coverage = len(unique_rec_cats) / len(total_cats_in_pool) * 100

print(f"\n  类别覆盖率: {len(unique_rec_cats)} / {len(total_cats_in_pool)} ({category_coverage:.1f}%)")

# Top推荐类别
rec_cat_counter = Counter(all_rec_categories)
print(f"\n  推荐中最常见的类别 Top 10:")
for cat, count in rec_cat_counter.most_common(10):
    print(f"    {cat:<30s} {count:4d}")

# 3b. 单用户推荐内多样性 (Intra-List Diversity)
print(f"\n  单用户推荐内多样性 (ILD):")
ild_scores = []

for recs_df in all_results:
    if len(recs_df) < 2:
        continue
    # 获取推荐餐厅在business_tfidf中的位置
    rec_positions = []
    for bid in recs_df['business_id']:
        mask = business_profiles['business_id'] == bid
        if mask.any():
            pos = business_profiles.index.get_loc(business_profiles[mask].index[0])
            rec_positions.append(pos)
    
    if len(rec_positions) < 2:
        continue
    
    # 计算推荐列表内部的平均不相似度
    rec_vectors = business_tfidf[rec_positions]
    pairwise_sim = cosine_similarity(rec_vectors)
    
    # 取上三角（排除对角线）
    n = len(rec_positions)
    mask_tri = np.triu(np.ones((n, n), dtype=bool), k=1)
    avg_dissimilarity = 1 - pairwise_sim[mask_tri].mean()
    ild_scores.append(avg_dissimilarity)

avg_ild = np.mean(ild_scores)
print(f"  平均 ILD: {avg_ild:.4f} (越接近1越多样)")
print(f"  ILD 范围: [{min(ild_scores):.4f}, {max(ild_scores):.4f}]")

# ========================================
# 21.5 评估摘要
# ========================================
print("\n" + "=" * 80)
print("📋 评估摘要")
print("=" * 80)

print(f"""
  ┌────────────────────────────────────────────┐
  │  评估用户数:        {valid_users:>6d}                 │
  │  推荐平均评分:      {rec_avg_stars:>6.2f} ★               │
  │  候选池平均评分:    {pool_avg_stars:>6.2f} ★               │
  │  评分提升:          {rec_avg_stars - pool_avg_stars:>+6.2f} ★               │
  │  平均相似度:        {rec_avg_similarity:>6.4f}                │
  │  类别覆盖率:        {category_coverage:>5.1f}%                │
  │  平均 ILD:          {avg_ild:>6.4f}                │
  │  vs 随机 p-value:   {p_value:>6.4f} {'✅' if p_value < 0.05 else '❌'}              │
  └────────────────────────────────────────────┘
""")

In [ ]:
print("=" * 80)
print("🎯 指标4: 留一法个性化评估 (Leave-One-Out)")
print("=" * 80)

hit_at_10 = 0
hit_at_20 = 0
mrr_scores = []
total_valid = 0

for uid in eval_user_ids:
    user_revs = food_reviews_df[food_reviews_df['user_id'] == uid]
    
    # 至少需要2条评论才能做留一法
    if len(user_revs) < 2:
        continue
    
    # 只对用户高评分的餐厅做测试（>=4★）
    high_rated = user_revs[user_revs['stars'] >= 4]
    if len(high_rated) == 0:
        continue
    
    for _, held_out in high_rated.iterrows():
        held_out_biz = held_out['business_id']
        
        # 检查这家餐厅是否在候选池中
        if held_out_biz not in business_profiles['business_id'].values:
            continue
        
        # 用剩余评论构建训练集
        train_revs = user_revs[user_revs['business_id'] != held_out_biz]
        
        # 临时替换food_reviews_df中该用户的评论
        temp_reviews = food_reviews_df[
            ~((food_reviews_df['user_id'] == uid) & 
              (food_reviews_df['business_id'] == held_out_biz))
        ]
        
        try:
            recs, _ = content_based_recommendations(
                user_id=uid,
                business_tfidf=business_tfidf,
                business_profiles=business_profiles,
                business_vectorizer=business_vectorizer,
                food_reviews_df=temp_reviews,
                top_k=20
            )
        except:
            continue
        
        if recs is None or len(recs) == 0:
            continue
        
        total_valid += 1
        rec_biz_ids = recs['business_id'].tolist()
        
        # Hit@10: 被隐藏的餐厅是否出现在Top10
        if held_out_biz in rec_biz_ids[:10]:
            hit_at_10 += 1
        
        # Hit@20
        if held_out_biz in rec_biz_ids[:20]:
            hit_at_20 += 1
        
        # MRR: 被隐藏的餐厅排在第几位
        if held_out_biz in rec_biz_ids:
            rank = rec_biz_ids.index(held_out_biz) + 1
            mrr_scores.append(1.0 / rank)
        else:
            mrr_scores.append(0.0)

# 计算指标
print(f"\n  评估样本数: {total_valid}")
print(f"\n  Hit@10:  {hit_at_10}/{total_valid} = {hit_at_10/max(total_valid,1)*100:.2f}%")
print(f"  Hit@20:  {hit_at_20}/{total_valid} = {hit_at_20/max(total_valid,1)*100:.2f}%")
print(f"  MRR:     {np.mean(mrr_scores):.4f}")

print(f"\n  解读:")
print(f"  - Hit@10: 用户真正喜欢的餐厅有 {hit_at_10/max(total_valid,1)*100:.1f}% 概率出现在Top10推荐中")
print(f"  - MRR: 平均排在第 {1/max(np.mean(mrr_scores),0.001):.1f} 位")

---

# 阶段八：扩展实验（可选）


## Step 30: 参数调优实验


In [ ]:
print("="*80)
print("🔧 Step 30: TF-IDF参数调优实验")
print("="*80)

# 30.1 不同max_features对推荐的影响
print("\n1️⃣ max_features参数影响")
print("-" * 60)

max_features_list = [500, 1000, 2000, 3000, 5000]
results = []

for max_feat in max_features_list:
    # 创建向量化器
    test_vectorizer = TfidfVectorizer(
        max_features=max_feat,
        min_df=2,
        max_df=0.7,
        stop_words='english'
    )

    # 向量化
    test_tfidf = test_vectorizer.fit_transform(
        business_profiles['combined_text'])

    # 计算推荐相似度
    user_text = ' '.join(user_reviews['text'].tolist())
    user_vec = test_vectorizer.transform([user_text])
    sims = cosine_similarity(user_vec, test_tfidf).flatten()

    results.append({
        'max_features': max_feat,
        'feature_dim': test_tfidf.shape[1],
        'top1_similarity': sims.max(),
        'avg_top5_similarity': np.sort(sims)[-5:].mean(),
        'sparsity': 1 - test_tfidf.nnz / (test_tfidf.shape[0] * test_tfidf.shape[1])
    })

results_df = pd.DataFrame(results)
print("\n参数影响结果:")
display(results_df)

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(results_df['max_features'],
             results_df['top1_similarity'], 'o-', label='Top 1')
axes[0].plot(results_df['max_features'],
             results_df['avg_top5_similarity'], 's-', label='Avg Top 5')
axes[0].set_xlabel('max_features')
axes[0].set_ylabel('Similarity Score')
axes[0].set_title('Similarity vs max_features')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(results_df['max_features'],
             results_df['sparsity'] * 100, 'o-', color='coral')
axes[1].set_xlabel('max_features')
axes[1].set_ylabel('Sparsity (%)')
axes[1].set_title('Matrix Sparsity vs max_features')
axes[1].grid(True)

plt.tight_layout()
plt.show()

---

# 实验总结

## 核心知识点回顾

### 1. TF-IDF 算法原理

- **TF (词频)**: 词在文档中出现的频率
- **IDF (逆文档频率)**: 词的稀有程度，log((N+1)/(ni+1))
- **TF-IDF**: TF × IDF，同时考虑词在文档内的重要性和在文档间的区分度

### 2. 基于内容的推荐流程

```
用户历史评论 → 文本向量化(TF-IDF) → 构建用户画像 → 计算相似度 → 推荐餐厅
餐厅评论聚合 → 文本向量化(TF-IDF) → 构建餐厅特征 → 计算相似度 → 排序推荐
```

### 3. 关键技术点

- **数据预处理**: JSON Lines 读取、文本清洗、停用词过滤
- **特征工程**: 评论聚合、类别融合、TF-IDF 向量化
- **相似度计算**: 余弦相似度最适合高维稀疏文本向量
- **冷启动处理**: 基于类别的推荐作为备选方案

## 实验收获

1. ✅ 手动实现了 TF-IDF 算法，理解其数学原理
2. ✅ 掌握了 sklearn TfidfVectorizer 的使用和参数调优
3. ✅ 构建了一个完整的基于内容的推荐系统
4. ✅ 学会了评估推荐系统的基本方法